In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/trigu006/Desktop/Projects/RAG-Evaluation-Platform/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

docs = splitter.split_text("This is a test document. It will be split into chunks based on the specified chunk size and overlap. The splitter will ensure that the chunks are created in a way that maintains the context of the original document. This is important for tasks such as information retrieval, where we want to preserve the meaning of the text while breaking it down into manageable pieces.")

In [3]:
docs

['This is a test document. It will be split into chunks based on the specified chunk size and overlap.',
 'size and overlap. The splitter will ensure that the chunks are created in a way that maintains the',
 'that maintains the context of the original document. This is important for tasks such as',
 'for tasks such as information retrieval, where we want to preserve the meaning of the text while',
 'of the text while breaking it down into manageable pieces.']

## Open the docs and concatenate them as text

#### Open and chunk docs

In [1]:
with open("docs/Northstar Analytics — 401(k) Summary Plan Description.md", "r") as file:
    text = file.read()

In [2]:
from pathlib import Path

directory_path = Path("/Users/trigu006/Desktop/Projects/RAG-Evaluation-Platform/docs")

text = ""

for file in directory_path.iterdir():
    print(file.name)
    with open(file, "r") as f:
        text += f.read()

Northstar Analytics — Parental and Family Leave Policy.md
Northstar Analytics — Employee Handbook.md
Northstar Analytics — 401(k) Summary Plan Description.md


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

headers_to_split_on = [
    ('#', 'document_title'),
    ('##', 'section_title'),
    ('###', 'subsection_title')
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)

/Users/trigu006/Desktop/Projects/RAG-Evaluation-Platform/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
markdown_docs = markdown_splitter.split_text(text)

In [5]:
markdown_docs

[Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content="# Northstar Analytics, Inc.  \n## Parental and Family Leave Policy  \n**Version:** 2.2\n**Effective Date:** March 1, 2026\n**Status:** Active  \nThis policy governs Northstar's company-provided parental leave benefit.  \nThis policy does not describe every leave entitlement that may exist under federal, state, or local law."),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Eligibility'}, page_content='## 1. Eligibility  \nFull-time employees become eligible for Northstar paid parental leave after completing **six continuous months of employment**.  \nPart-time employees, temporary employees, and independent contractors are not eligible for company-provided paid parental leave.'),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '2. Qualifying Events'}, page_content="## 2. Qualif

In [6]:
chunker = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

In [7]:
chunks = chunker.split_documents(markdown_docs)

In [8]:
chunks

[Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content='# Northstar Analytics, Inc.  \n## Parental and Family Leave Policy  \n**Version:** 2.2'),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content='**Version:** 2.2\n**Effective Date:** March 1, 2026\n**Status:** Active'),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content="This policy governs Northstar's company-provided parental leave benefit."),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content='This policy does not describe every leave entitlement that may exist under federal, state, or local'),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Polic

In [9]:
from src.enterprise_policy_rag.chunking.base import markdown_chunker

In [10]:
docs = markdown_chunker(text, chunk_size=150, chunk_overlap=20)

In [11]:
docs

[Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content='# Northstar Analytics, Inc.  \n## Parental and Family Leave Policy  \n**Version:** 2.2\n**Effective Date:** March 1, 2026\n**Status:** Active'),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content="This policy governs Northstar's company-provided parental leave benefit."),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': 'Parental and Family Leave Policy'}, page_content='This policy does not describe every leave entitlement that may exist under federal, state, or local law.'),
 Document(metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Eligibility'}, page_content='## 1. Eligibility  \nFull-time employees become eligible for Northstar paid parental leave after completing **six continuous months of employment**.'

#### Initiate embedding model and create vector data store

In [12]:
from src.enterprise_policy_rag.embeddings.ollama_embeddings import get_embedding_model

In [13]:
embedding_model = get_embedding_model()

In [14]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(docs, embedding_model)

In [15]:
from src.enterprise_policy_rag.retrieval.vector_store import create_vector_store

vector_store = create_vector_store(docs, embedding_model, reset=True)

#### Sample questions

In [17]:
vector_store.similarity_search("How many months of parental leave do we have?")

[Document(id='d852867a-4de2-4e90-97db-18c8ed6bd8e8', metadata={'section_title': '3. Duration', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 3. Duration  \nEligible employees may receive up to **eight weeks of paid parental leave** for a qualifying event.'),
 Document(id='8a195eea-7c29-4f5c-858b-36c6dd71575a', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '8. Multiple Qualifying Events'}, page_content='Paid parental leave is limited to **eight weeks in a rolling 12-month period**, regardless of the number of qualifying events occurring during that'),
 Document(id='405a90b6-50a3-446f-b394-c1df2c78d809', metadata={'section_title': '7. Holidays During Parental Leave', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 7. Holidays During Parental Leave'),
 Document(id='affc2f95-8c57-49cc-83a5-3b1c6d477ab8', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '4. Compensation'}, page_content="## 4. Compensati

In [7]:
from src.enterprise_policy_rag.retrieval.retriever import create_retriever

In [24]:
retriever = create_retriever(vector_store)

In [26]:
foo = retriever.invoke("How many months of parental leave do we have?")
foo

[Document(id='99b5058b-3c2d-400f-828e-0527b5e07bf7', metadata={'section_title': '3. Duration', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 3. Duration  \nEligible employees may receive up to **eight weeks of paid parental leave** for a qualifying event.'),
 Document(id='63709b35-ce11-418d-9f0a-1d942d4d3e6f', metadata={'section_title': '8. Multiple Qualifying Events', 'document_title': 'Northstar Analytics, Inc.'}, page_content='Paid parental leave is limited to **eight weeks in a rolling 12-month period**, regardless of the number of qualifying events occurring during that'),
 Document(id='d13cd86d-faaf-4d7f-be36-8a96ed63e170', metadata={'section_title': '7. Holidays During Parental Leave', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 7. Holidays During Parental Leave'),
 Document(id='3a29b016-6f1e-439e-85a3-ba62ed1671f8', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '4. Compensation'}, page_content="## 4. Compensati

In [43]:
foo

for document in foo:
    if '3. Duration' in document.metadata.get('section_title', ''):
        print(document.metadata.get('section_title', ''))
    else:
        print("No matching section found in this document.")

3. Duration
No matching section found in this document.
No matching section found in this document.


In [37]:
retriever.invoke("If I have five years of tenure, how many PTO days do I get per year?")

[Document(id='3009405a-e602-4b47-ad22-214d7d66a561', metadata={'section_title': '2. Paid Time Off', 'document_title': 'Northstar Analytics, Inc.'}, page_content='Annual PTO entitlement is based on completed years of continuous service as of January 1 of each calendar year:  \n| Completed Service | Annual PTO |'),
 Document(id='4d944ab6-28d4-4a3a-89f6-b28564ec3894', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '2. Paid Time Off'}, page_content='Employees may carry over a maximum of **five unused PTO days** from one calendar year into the next. PTO above the five-day carryover limit is'),
 Document(id='b854b103-fa2d-4922-ae04-9310fea1b68d', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '2. Paid Time Off'}, page_content='New full-time employees receive prorated PTO during their first calendar year based on their hire date.')]

In [27]:
retriever.invoke("When are employees eligible to participate in the 401(k) plan?")


[Document(id='5b2aa50a-a9f6-4260-82db-0b30b3708377', metadata={'section_title': '9. Relationship to Other Northstar Documents', 'document_title': 'Northstar Analytics, Inc.'}, page_content='Information about the 401(k) Plan that appears in an employee handbook, onboarding guide, presentation, email, or other informal communication does'),
 Document(id='a9b35874-19d5-4929-97d5-9e285e0fb5a9', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Employee Eligibility'}, page_content='## 1. Employee Eligibility  \nFull-time and part-time employees become eligible to participate after completing **90 days of service**.'),
 Document(id='7de5655d-dbe0-407d-846d-bf8129871767', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Employee Eligibility'}, page_content='An eligible employee may begin making elective contributions on the first payroll period administratively practicable following completion of the')]

In [39]:
retriever.invoke("How many company holidays are there each year?")

[Document(id='ac1a6bf0-2923-4bad-a003-694946b43b0c', metadata={'section_title': '4. Company Holidays', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 4. Company Holidays  \nNorthstar observes ten company holidays each calendar year.'),
 Document(id='12db06bd-33fb-4e82-9846-de58f72948ed', metadata={'section_title': '4. Company Holidays', 'document_title': 'Northstar Analytics, Inc.'}, page_content='In addition, full-time employees receive **two floating holidays** per calendar year.  \nFloating holidays:'),
 Document(id='f7251622-0952-4598-a0e2-2fe636d8a13c', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '7. Holidays During Parental Leave'}, page_content='## 7. Holidays During Parental Leave')]

In [29]:
for i, doc in enumerate(results, start = 1):
    print(f"Document {i}:")
    print(doc.metadata)
    print(doc.page_content)
    print("\n---\n")

Document 1:
{'document_title': 'Northstar Analytics, Inc.', 'section_title': '9. Relationship to Other Northstar Documents'}
Information about the 401(k) Plan that appears in an employee handbook, onboarding guide, presentation, email, or other informal communication does

---

Document 2:
{'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Employee Eligibility'}
## 1. Employee Eligibility  
Full-time and part-time employees become eligible to participate after completing **90 days of service**.

---

Document 3:
{'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Employee Eligibility'}
An eligible employee may begin making elective contributions on the first payroll period administratively practicable following completion of the

---



## View contents of Chroma database

In [31]:
import chromadb

client = chromadb.PersistentClient(path="./data/chroma")

collection = client.get_collection("employee_handbook")

collection.peek()

{'ids': ['e8bc931c-08a5-41d2-b48f-01fed87af16a',
  '2ccc43c2-082a-4531-a06b-3949b2873256',
  '9b581431-d418-4631-8aa7-37be38c76140',
  'd43bb171-0dfd-4b39-a3a9-55c6472cf098',
  '35904e78-4719-46a5-bd72-13b2eb1aa320',
  'de571840-131c-43bf-a824-686ddecf060e',
  '923bc623-6025-4908-b3fc-36b7d2aa647c',
  'ab5fb611-cbce-4bd4-87dc-d393b9774530',
  '0d969b26-f0fd-4f00-924f-9b4eac0ff554',
  '117fc052-4207-4322-b2ef-037489b894d0'],
 'embeddings': array([[-0.08876898,  0.04136869,  0.03322151, ...,  0.0900713 ,
          0.00705203, -0.06322747],
        [-0.0662201 ,  0.0583501 ,  0.02463884, ...,  0.03856012,
          0.03007659, -0.06558982],
        [-0.05628746,  0.03506787,  0.07794673, ..., -0.00244526,
          0.00336673, -0.04181143],
        ...,
        [-0.03441419,  0.06741494, -0.00673583, ..., -0.01715781,
          0.03975511, -0.00458936],
        [-0.07859083,  0.08848472,  0.03281007, ..., -0.00368959,
          0.00739086, -0.02032197],
        [-0.0286042 ,  0.04297914, 

In [33]:
import pandas as pd

results = collection.get(
    include = ['documents', 'metadatas', 'embeddings']
)

rows = []

for i, chunk_id in enumerate(results["ids"]):
    vector = results["embeddings"][i]

    rows.append({
        "id": chunk_id,
        "document": results["documents"][i],
        "metadata": results["metadatas"][i],
        "embedding_dimensions": len(vector),
        "vector_preview": vector[:8],
    })

df = pd.DataFrame(rows)

display(df)

,id,document,metadata,embedding_dimensions,vector_preview
0,e8bc931c-08a5-41d2-b48f-01fed87af16a,"# Northstar Analytics, Inc. \n## Parental and...","{'document_title': 'Northstar Analytics, Inc.'...",768,"[-0.08876898139715195, 0.04136868566274643, 0...."
1,2ccc43c2-082a-4531-a06b-3949b2873256,This policy governs Northstar's company-provid...,"{'document_title': 'Northstar Analytics, Inc.'...",768,"[-0.06622009724378586, 0.058350104838609695, 0..."
2,9b581431-d418-4631-8aa7-37be38c76140,This policy does not describe every leave enti...,"{'document_title': 'Northstar Analytics, Inc.'...",768,"[-0.056287456303834915, 0.03506787121295929, 0..."
3,d43bb171-0dfd-4b39-a3a9-55c6472cf098,## 1. Eligibility \nFull-time employees becom...,"{'section_title': '1. Eligibility', 'document_...",768,"[-0.07189580798149109, 0.05202243849635124, 0...."
4,35904e78-4719-46a5-bd72-13b2eb1aa320,"Part-time employees, temporary employees, and ...","{'section_title': '1. Eligibility', 'document_...",768,"[-0.07009243965148926, 0.02994685247540474, 0...."
...,...,...,...,...,...
127,39885c67-0229-4464-adce-d593bd6f32aa,If another Northstar policy provides different...,{'section_title': '9. Relationship to Other No...,768,"[-0.06103772297501564, 0.016987059265375137, 0..."
128,6fdf478d-bb9d-4303-90c3-0736f6d04ff8,"- loans; or\n- distributions, \nemployees sho...","{'document_title': 'Northstar Analytics, Inc.'...",768,"[-0.07696747034788132, 0.08449826389551163, 0...."
129,e1a6052b-40af-4987-8f28-a5304f73401b,## 10. External Legal Limits \nFederal contri...,"{'section_title': '10. External Legal Limits',...",768,"[-0.03595457598567009, 0.0013608960434794426, ..."
130,1659c2ed-e2a3-4564-8164-9bf585c08b69,Questions regarding the **current federal annu...,"{'section_title': '10. External Legal Limits',...",768,"[-0.06726307421922684, -0.00942462868988514, 0..."


## Test evaluations

#### Test different configurations

In [16]:
from src.enterprise_policy_rag.evaluation.retriever_config import RETRIEVER_CONFIGS
from src.enterprise_policy_rag.retrieval.retriever import create_retriever

ret = create_retriever(vector_store, **RETRIEVER_CONFIGS[0]['params'])

In [17]:
CONFIG = {
        "name": "similarity_k1",
        "params": {
            "search_type": "similarity",
            "k": 3,
        }
    }

testy = create_retriever(vector_store, **CONFIG['params'])

In [18]:
from src.enterprise_policy_rag.evaluation.run_retriever_evaluations import evaluate_retriever_configs

results = evaluate_retriever_configs(vector_store)

In [19]:
for config in RETRIEVER_CONFIGS:
    print(config)

{'name': 'similarity_k1', 'params': {'search_type': 'similarity', 'k': 1}}
{'name': 'similarity_k3', 'params': {'search_type': 'similarity', 'k': 3}}
{'name': 'similarity_k4', 'params': {'search_type': 'similarity', 'k': 4}}
{'name': 'similarity_k5', 'params': {'search_type': 'similarity', 'k': 5}}
{'name': 'similarity_k5', 'params': {'search_type': 'similarity', 'k': 5}}
{'name': 'mmr_k6', 'params': {'search_type': 'mmr', 'k': 6}}


In [20]:
results

{'similarity_k1': {'overall_scores': {'hit_at_k': 0.7,
   'mrr': 0.7,
   'semantic_answer_sufficiency': 0.1},
  'records': [{'question_id': 'pto_001',
    'hit_at_k_score': 1,
    'mrr_score': 1.0,
    'semantic_answer_sufficiency_score': np.float64(0.5057379613457956),
    'hit': 0},
   {'question_id': '401k_002',
    'hit_at_k_score': 0,
    'mrr_score': 0,
    'semantic_answer_sufficiency_score': np.float64(0.3366196052534246),
    'hit': 0},
   {'question_id': 'sick_003',
    'hit_at_k_score': 1,
    'mrr_score': 1.0,
    'semantic_answer_sufficiency_score': np.float64(0.4063837039838582),
    'hit': 0},
   {'question_id': 'pto_004',
    'hit_at_k_score': 1,
    'mrr_score': 1.0,
    'semantic_answer_sufficiency_score': np.float64(0.2853423532678224),
    'hit': 0},
   {'question_id': 'holidays_005',
    'hit_at_k_score': 1,
    'mrr_score': 1.0,
    'semantic_answer_sufficiency_score': np.float64(0.40860359126711326),
    'hit': 0},
   {'question_id': 'pto_006',
    'hit_at_k_scor

#### Test questions and select the best configuration

In [21]:
from src.enterprise_policy_rag.evaluation.select_best_config import select_best_config

best_config, deltas = select_best_config(results)
print(best_config)

{'name': 'mmr_k6', 'scores': {'hit_at_k': 0.85, 'mrr': 0.7666666666666667, 'semantic_answer_sufficiency': 0.15}, 'individual_scores': {'hit_at_k': 0.85, 'mrr': 0.7666666666666667, 'semantic_answer_sufficiency': 0.15}, 'records': [{'question_id': 'pto_001', 'hit_at_k_score': 1, 'mrr_score': 1.0, 'semantic_answer_sufficiency_score': np.float64(0.5057379613457956), 'hit': 0}, {'question_id': '401k_002', 'hit_at_k_score': 1, 'mrr_score': 0.5, 'semantic_answer_sufficiency_score': np.float64(0.6412266510459049), 'hit': 0}, {'question_id': 'sick_003', 'hit_at_k_score': 1, 'mrr_score': 1.0, 'semantic_answer_sufficiency_score': np.float64(0.4063837039838582), 'hit': 0}, {'question_id': 'pto_004', 'hit_at_k_score': 1, 'mrr_score': 1.0, 'semantic_answer_sufficiency_score': np.float64(0.4609124379810909), 'hit': 0}, {'question_id': 'holidays_005', 'hit_at_k_score': 1, 'mrr_score': 1.0, 'semantic_answer_sufficiency_score': np.float64(0.40860359126711315), 'hit': 0}, {'question_id': 'pto_006', 'hit_

In [22]:
deltas

{'similarity_k1': {'absolute': {'hit_at_k': 0.15000000000000002,
   'mrr': 0.06666666666666676,
   'semantic_answer_sufficiency': 0.04999999999999999,
   'k_difference': 5},
  'headroom_normalized': {'hit_at_k': 0.5,
   'mrr': 0.22222222222222252,
   'semantic_answer_sufficiency': 0.05555555555555554,
   'k_difference': 5.0}},
 'similarity_k3': {'absolute': {'hit_at_k': 0.09999999999999998,
   'mrr': 0.04166666666666674,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 3},
  'headroom_normalized': {'hit_at_k': 0.3999999999999999,
   'mrr': 0.15151515151515177,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 1.0}},
 'similarity_k4': {'absolute': {'hit_at_k': 0.09999999999999998,
   'mrr': 0.04166666666666674,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 2},
  'headroom_normalized': {'hit_at_k': 0.3999999999999999,
   'mrr': 0.15151515151515177,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 0.5}},
 'similarity_k5': {'absolute': {'hit_at_k': 0.0

In [23]:
deltas

{'similarity_k1': {'absolute': {'hit_at_k': 0.15000000000000002,
   'mrr': 0.06666666666666676,
   'semantic_answer_sufficiency': 0.04999999999999999,
   'k_difference': 5},
  'headroom_normalized': {'hit_at_k': 0.5,
   'mrr': 0.22222222222222252,
   'semantic_answer_sufficiency': 0.05555555555555554,
   'k_difference': 5.0}},
 'similarity_k3': {'absolute': {'hit_at_k': 0.09999999999999998,
   'mrr': 0.04166666666666674,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 3},
  'headroom_normalized': {'hit_at_k': 0.3999999999999999,
   'mrr': 0.15151515151515177,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 1.0}},
 'similarity_k4': {'absolute': {'hit_at_k': 0.09999999999999998,
   'mrr': 0.04166666666666674,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 2},
  'headroom_normalized': {'hit_at_k': 0.3999999999999999,
   'mrr': 0.15151515151515177,
   'semantic_answer_sufficiency': 0.0,
   'k_difference': 0.5}},
 'similarity_k5': {'absolute': {'hit_at_k': 0.0

In [24]:
for config_name, deltas_config in deltas.items():
    print(config_name)

similarity_k1
similarity_k3
similarity_k4
similarity_k5


In [25]:
best_config

{'name': 'mmr_k6',
 'scores': {'hit_at_k': 0.85,
  'mrr': 0.7666666666666667,
  'semantic_answer_sufficiency': 0.15},
 'individual_scores': {'hit_at_k': 0.85,
  'mrr': 0.7666666666666667,
  'semantic_answer_sufficiency': 0.15},
 'records': [{'question_id': 'pto_001',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.5057379613457956),
   'hit': 0},
  {'question_id': '401k_002',
   'hit_at_k_score': 1,
   'mrr_score': 0.5,
   'semantic_answer_sufficiency_score': np.float64(0.6412266510459049),
   'hit': 0},
  {'question_id': 'sick_003',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.4063837039838582),
   'hit': 0},
  {'question_id': 'pto_004',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.4609124379810909),
   'hit': 0},
  {'question_id': 'holidays_005',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': 

In [26]:
import numpy as np

eligible_configs = [{'name': 'similarity_k5',
 'scores': {'hit_at_k': 1.0,
  'mrr': 0.876470588235294,
  'semantic_answer_sufficiency': 0.17647058823529413},
 'individual_scores': {'hit_at_k': 1.0,
  'mrr': 0.876470588235294,
  'semantic_answer_sufficiency': 0.17647058823529413},
 'records': [{'question_id': 'pto_001',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.5057221977340789),
   'hit': 0},
  {'question_id': '401k_002',
   'hit_at_k_score': 1,
   'mrr_score': 0.5,
   'semantic_answer_sufficiency_score': np.float64(0.6409139905036544),
   'hit': 0},
  {'question_id': 'sick_003',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.4063921462133845),
   'hit': 0},
  {'question_id': 'pto_004',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.46089824188523965),
   'hit': 0},
  {'question_id': 'holidays_005',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.408595626886389),
   'hit': 0},
  {'question_id': 'pto_006',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.4905478706185282),
   'hit': 0},
  {'question_id': 'bereavement_007',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.5273607130326387),
   'hit': 0},
  {'question_id': 'tuition_008',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.6657330993140134),
   'hit': 0},
  {'question_id': 'tuition_009',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.7065145783758379),
   'hit': 0},
  {'question_id': 'holidays_010',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.803639315472788),
   'hit': 1},
  {'question_id': '401k_011',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.8848230452888965),
   'hit': 1},
  {'question_id': '401k_012',
   'hit_at_k_score': 1,
   'mrr_score': 0.2,
   'semantic_answer_sufficiency_score': np.float64(0.5255774946941718),
   'hit': 0},
  {'question_id': '401k_013',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.9707390427814194),
   'hit': 1},
  {'question_id': '401k_014',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.7279231867140477),
   'hit': 0},
  {'question_id': '401k_015',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.540916282155227),
   'hit': 0},
  {'question_id': '401k_016',
   'hit_at_k_score': 1,
   'mrr_score': 0.2,
   'semantic_answer_sufficiency_score': np.float64(0.6255795929272069),
   'hit': 0},
  {'question_id': '401k_017',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.7437592460597452),
   'hit': 0}]}]

if eligible_configs:
        for i in range(len(eligible_configs)):
            if i == 0:
                best_eligible_config = eligible_configs[i]
            else:
                if deltas[best_eligible_config['name']]['absolute']['k_difference'] < deltas[eligible_configs[i]['name']]['absolute']['k_difference']:
                    best_eligible_config = eligible_configs[i]
        best_config = best_eligible_config

best_config

{'name': 'similarity_k5',
 'scores': {'hit_at_k': 1.0,
  'mrr': 0.876470588235294,
  'semantic_answer_sufficiency': 0.17647058823529413},
 'individual_scores': {'hit_at_k': 1.0,
  'mrr': 0.876470588235294,
  'semantic_answer_sufficiency': 0.17647058823529413},
 'records': [{'question_id': 'pto_001',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.5057221977340789),
   'hit': 0},
  {'question_id': '401k_002',
   'hit_at_k_score': 1,
   'mrr_score': 0.5,
   'semantic_answer_sufficiency_score': np.float64(0.6409139905036544),
   'hit': 0},
  {'question_id': 'sick_003',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.4063921462133845),
   'hit': 0},
  {'question_id': 'pto_004',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'semantic_answer_sufficiency_score': np.float64(0.46089824188523965),
   'hit': 0},
  {'question_id': 'holidays_005',
   'hit_at_k_score': 1,
   'mrr_score': 1.0,
   'se

#### Initiate the generator (i.e., LLM that will answer users' questions)

*October 3, 2026*

In [27]:
from src.enterprise_policy_rag.generator.create_generator import create_generator

# Instantiate the generator (LLM)
llm = create_generator(model="phi4-mini:3.8b")

best_config_name = best_config['name']
best_config = None

# Instantiate the retriever
for config in RETRIEVER_CONFIGS:
    if config['name'] == best_config_name:
        best_config = config
        break

retriever = create_retriever(vector_store, **best_config['params'])

In [28]:
from src.enterprise_policy_rag.generator.generate_answer import generate_answer

sections = retriever.invoke("When are employees eligible to participate in the 401(k) plan?")

response = generate_answer(sections, llm, "When are employees eligible to participate in the 401(k) plan?")

In [29]:
response['answerable']

True

In [30]:
sections = retriever.invoke("When are employees eligible to participate in the 401(k) plan?")

In [31]:
sections

[Document(id='40610e5e-38b4-469c-8ba3-016e0c90e4ad', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '9. Relationship to Other Northstar Documents'}, page_content='Information about the 401(k) Plan that appears in an employee handbook, onboarding guide, presentation, email, or other informal communication does'),
 Document(id='5b864a3a-1de7-4009-94f4-337781bd4b66', metadata={'section_title': '1. Employee Eligibility', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 1. Employee Eligibility  \nFull-time and part-time employees become eligible to participate after completing **90 days of service**.'),
 Document(id='5afe361b-cd0e-47d0-92ba-e68fd7b1f84a', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '1. Employee Eligibility'}, page_content='An eligible employee may begin making elective contributions on the first payroll period administratively practicable following completion of the'),
 Document(id='91234e4c-4bd1-46ab-

In [32]:
sections[0].model_dump_json()

'{"id":"40610e5e-38b4-469c-8ba3-016e0c90e4ad","metadata":{"document_title":"Northstar Analytics, Inc.","section_title":"9. Relationship to Other Northstar Documents"},"page_content":"Information about the 401(k) Plan that appears in an employee handbook, onboarding guide, presentation, email, or other informal communication does","type":"Document"}'

In [33]:
# sections_string = []

# for section in sections:
#     sections_string.append(str(section.model_dump_json()))

# sections_string = "\n\n".join(sections_string)
# display(sections_string)

In [34]:
sections_string = []

sections_string = "\n\n".join(
    f"Section: {section.metadata.get('section_title', 'Unknown')}\n"
    f"{section.page_content}"
    for section in sections
)

sections_string

'Section: 9. Relationship to Other Northstar Documents\nInformation about the 401(k) Plan that appears in an employee handbook, onboarding guide, presentation, email, or other informal communication does\n\nSection: 1. Employee Eligibility\n## 1. Employee Eligibility  \nFull-time and part-time employees become eligible to participate after completing **90 days of service**.\n\nSection: 1. Employee Eligibility\nAn eligible employee may begin making elective contributions on the first payroll period administratively practicable following completion of the\n\nSection: 10. Policy Authority\nNorthstar Analytics 401(k) Plan\n\nSection: 1. Employee Eligibility\nTemporary employees and independent contractors are not eligible to participate.'

#### Test generator

*October 4, 2026*

In [35]:
from src.enterprise_policy_rag.generator.run_generator_evaluations import run_generator_evaluations
from src.enterprise_policy_rag.evaluation.questions import TEST_CASES

results, scores = run_generator_evaluations(retriever, llm, TEST_CASES)

In [36]:
results

[{'id': 'pto_001',
  'question': 'How many months of parental leave do we have?',
  'answer': 'Eight months',
  'predicted_answerable': True,
  'expected_answerable': True,
  'sections': [Document(id='38388a85-77d7-40f1-9fc7-1d0691d6ff34', metadata={'section_title': '3. Duration', 'document_title': 'Northstar Analytics, Inc.'}, page_content='## 3. Duration  \nEligible employees may receive up to **eight weeks of paid parental leave** for a qualifying event.'),
   Document(id='3713afdc-e3ec-4c69-b6ca-0fa4eaf1354b', metadata={'section_title': '8. Multiple Qualifying Events', 'document_title': 'Northstar Analytics, Inc.'}, page_content='Paid parental leave is limited to **eight weeks in a rolling 12-month period**, regardless of the number of qualifying events occurring during that'),
   Document(id='8957073b-7e78-4671-9683-54b4e57bc71c', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '7. Holidays During Parental Leave'}, page_content='## 7. Holidays During Pare

In [37]:
scores

{'answerability_accuracy': 1.0}

#### Test orchestrator

*October 16, 2026*

In [38]:
from src.enterprise_policy_rag.orchestration.run_rag_evaluations import run_rag_evaluations
from src.enterprise_policy_rag.evaluation.questions import TEST_CASES

generator = create_generator(model="phi4-mini:3.8b")

evals = run_rag_evaluations(retriever, generator, TEST_CASES, embedding_model)

In [39]:
evals

[{'id': 'pto_001',
  'question': 'How many months of parental leave do we have?',
  'answer': 'Eight months',
  'predicted_answerable': True,
  'expected_answerable': True,
  'best_score': np.float64(0.5057379613457956),
  'hit': 0,
  'hit_at_k_score': 1,
  'reciprocal_rank': 1.0,
  'sections': [Document(id='38388a85-77d7-40f1-9fc7-1d0691d6ff34', metadata={'document_title': 'Northstar Analytics, Inc.', 'section_title': '3. Duration'}, page_content='## 3. Duration  \nEligible employees may receive up to **eight weeks of paid parental leave** for a qualifying event.'),
   Document(id='3713afdc-e3ec-4c69-b6ca-0fa4eaf1354b', metadata={'section_title': '8. Multiple Qualifying Events', 'document_title': 'Northstar Analytics, Inc.'}, page_content='Paid parental leave is limited to **eight weeks in a rolling 12-month period**, regardless of the number of qualifying events occurring during that'),
   Document(id='8957073b-7e78-4671-9683-54b4e57bc71c', metadata={'section_title': '7. Holidays Dur